# Assignment 2

## Imports

In [20]:
import scipy.io
import numpy as np
import wandb
import tqdm
import torch
import torchvision.io as io
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import Adam

In [21]:
wandb.login()

True

In [22]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Data

In [23]:
ids = scipy.io.loadmat('data/setid.mat')
# -1 because of 1 as starting index
trnid = ids['trnid'].flatten() - 1
valid = ids['valid'].flatten() - 1
tstid = ids['tstid'].flatten() - 1

In [24]:
path = 'data/102flowers/jpg/image_'
resize = transforms.Resize((224, 224))

X = torch.stack([resize(io.read_image(path + str(i).zfill(5) + '.jpg')) for i in range(1, 8190)])
X = X.float() / 255.0 # normalization

In [25]:
X_train = X[trnid]
X_val = X[valid]
X_test = X[tstid]

In [26]:
y = torch.tensor(scipy.io.loadmat('data/imagelabels.mat')['labels'], dtype=torch.long).flatten() - 1
y_train = y[trnid]
y_val = y[valid]
y_test = y[tstid]

In [27]:
train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

In [28]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

In [ ]:
train_ds_10 = torch.utils.data.Subset(full_train_ds, torch.randperm(n_train)[:n_train // 10])
val_ds_10 = torch.utils.data.Subset(full_val_ds, torch.randperm(n_val)[:n_val // 10])

train_loader_10 = DataLoader(train_ds_10, batch_size=32, shuffle=True)
val_loader_10 = DataLoader(val_ds_10, batch_size=32)

## Task 0

In [29]:
project = 'ADM-List2-T0'

lr = 1e-4
weight_decay = 1e-4
epochs = 50

config = {
    'epochs': epochs,
    'lr': lr,
    'weight_decay': weight_decay,
    'train_sample_count': len(train_ds)
}

In [30]:
class ExampleCNN(nn.Module):
    def __init__(self, num_classes=102):
        super(ExampleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Fully connected layers
        self.c = 56*56*16
        self.fc1 = nn.Linear(self.c, 1024) # Flatten (512 * 7 * 7) features
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(-1, self.c) # Flatten the tensor
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x) # logits for all classes
        return x

In [31]:
model = ExampleCNN().to(device)
optimizer = Adam(lr=lr, weight_decay=weight_decay, params=model.parameters())
loss = nn.CrossEntropyLoss()

In [35]:
with wandb.init(project=project, config=config) as run:
    for epoch in tqdm.trange(epochs):
        total_train_loss = 0
        total_train_correct = 0
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()

            preds = model(X)
            running_loss = loss(preds, y)
            total_train_loss += running_loss.item()
            total_train_correct += (preds.argmax(dim=1) == y).sum().item()
            running_loss.backward()
            optimizer.step()

        total_val_loss = 0
        total_val_correct = 0
        model.eval()
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                preds = model(X)
                total_val_loss += loss(preds, y).item()
                total_val_correct += (preds.argmax(dim=1) == y).sum().item()
        wandb.log({
            'train_loss': total_train_loss / len(train_loader),
            'val_loss': total_val_loss / len(val_loader),
            'train_acc': total_train_correct / len(train_ds),
            'val_acc': total_val_correct / len(val_ds),
            'epoch': epoch
        })


  0%|          | 0/50 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/tmp/ipykernel_31800/2387339684.py", line 6, in <module>
    for X, y in train_loader:
                ^^^^^^^^^^^^
  File "/home/michal/PycharmProjects/Artificial-Inteligence/.venv/lib64/python3.13/site-packages/torch/utils/data/dataloader.py", line 732, in __next__
    data = self._next_data()
  File "/home/michal/PycharmProjects/Artificial-Inteligence/.venv/lib64/python3.13/site-packages/torch/utils/data/dataloader.py", line 788, in _next_data
    data = self._dataset_fetcher.fetch(index)  # may raise StopIteration
  File "/home/michal/PycharmProjects/Artificial-Inteligence/.venv/lib64/python3.13/site-packages/torch/utils/data/_utils/fetch.py", line 55, in fetch
    return self.collate_fn(data)
           ~~~~~~~~~~~~~~~^^^^^^
  File "/home/michal/PycharmProjects/Artificial-Inteligence/.venv/lib64/python3.13/site-packages/torch/utils/data/_utils/collate.py", line 398, in default_collate
    return coll

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


### 10% of samples